In [11]:
import torch

B = 16
T = 9
S = 10
F = 768

sequence = torch.randn(B, T, S)
sequence = sequence.moveaxis(1, 2).unsqueeze(-1) # B, S, T, 1

def positional_encoding(
        d_model: int,
        t: torch.Tensor,
    ) -> torch.Tensor:
    """
    Args:
    - d_model: int
    - t: torch.Tensor (shape [batch_size, sequence_length])

    Returns: torch.Tensor (shape [batch_size, sequence_length, d_model])
    """
    inv_freq = 1.0 / (
        10000
        ** (torch.arange(0, d_model, 2, device=t.device) / d_model)
    )
    # Ensure `t` has shape [batch_size, sequence_length, 1]
    t = t.unsqueeze(-1)  # Shape [batch_size, sequence_length, 1]
    pos_enc_a = torch.sin(t * inv_freq)  # Shape [batch_size, sequence_length, d_model // 2]
    pos_enc_b = torch.cos(t * inv_freq)  # Shape [batch_size, sequence_length, d_model // 2]
    pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)  # Shape [batch_size, sequence_length, d_model]
    return pos_enc

embedders = torch.nn.ModuleList(
    torch.nn.Sequential(
        torch.nn.Linear(1, F),
        torch.nn.BatchNorm1d(T),
        torch.nn.ReLU(),
        torch.nn.Linear(F, F),
        torch.nn.BatchNorm1d(T),
        torch.nn.ReLU(),
    ) # (B, S, T, 1) x (1, F) = (B, S, T, F)
 for _ in range(S))

# Featurize each source independently
features = []
for si in range(S):
    s = sequence[:, si, :, :] # (B, T, 1)

    featurizer_si = embedders[si]
    f = featurizer_si(s) # (B, T, 1) x (1, F) = (B, T, F)

    ts = torch.range(0, T-1, device=s.device)
    pos_encodings = positional_encoding(F, ts) # (T, F)
    pos_encodings = pos_encodings.unsqueeze(0) # (1, T, F) || (B, T, F)
    f = f + pos_encodings # (B, T, F)
    features.append(f)

features = torch.stack(features, dim=1) # (B, S, T, F)

combined = features.reshape(B, S*T, F) # (B, S*T, F)

combiner = torch.nn.Transformer(
    d_model=F, 
    nhead=8, 
    num_encoder_layers=8, 
    num_decoder_layers=8,
    batch_first=True
)

result = combiner(combined[:, -T:, :],combined[:, -T:, :])
print(result)

C:\Users\ppaur\AppData\Local\Temp\ipykernel_11332\1815031968.py:52: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  ts = torch.range(0, T-1, device=s.device)


tensor([[[ 0.1021,  0.8348, -0.2083,  ..., -0.6730, -0.6386,  0.0982],
         [ 0.3700, -0.1940, -0.2856,  ...,  0.2583, -0.5487, -0.2603],
         [-0.6269,  0.1872, -0.2536,  ...,  0.1556,  0.0866, -0.6755],
         ...,
         [-0.5520,  0.5808, -1.2233,  ...,  0.3914,  0.4481, -1.0943],
         [-0.9760, -0.2017, -0.2330,  ...,  0.5601, -0.4262,  0.6605],
         [-0.6711,  0.1513, -0.0081,  ...,  0.8762, -0.0956, -0.6585]],

        [[ 0.8296,  0.5034, -0.6561,  ...,  0.4927,  0.2782,  0.5676],
         [ 0.1057, -0.0705,  0.8525,  ...,  0.4396, -0.8791, -0.2164],
         [ 0.8367,  0.6939, -0.8650,  ...,  0.5711, -0.3370, -0.6368],
         ...,
         [ 0.9358, -0.1562, -0.6421,  ...,  0.0627, -0.2659, -0.4158],
         [ 1.2464,  0.0873, -0.7262,  ...,  0.5118, -0.3563,  0.4455],
         [ 0.2298,  0.8453, -0.8612,  ...,  0.3302, -0.5108, -0.2094]],

        [[ 0.3352,  1.8863,  0.2601,  ...,  0.0715,  0.1654, -0.3558],
         [ 0.4568,  1.5217, -0.4187,  ...,  0

In [15]:
import torch

arr = torch.Tensor([[1,1,1],[2,2,2]])

print(arr)

arr = arr.reshape(2*3)
print(arr)

tensor([[1., 1., 1.],
        [2., 2., 2.]])
tensor([1., 1., 1., 2., 2., 2.])


In [11]:
import numpy as np

arr_1 = np.array([1,-1,1,-1,1,-1])
arr = np.array([1,2,3,4,5,6,7])
arr_2 = np.array([1,2,3,4,5,6,7])

print(arr_1)
print(arr)

[ 1 -1  1 -1  1 -1]
[1 2 3 4 5 6 7]


In [13]:
print(arr[1:])
print(arr_2[:-1])
print(arr_1)

[2 3 4 5 6 7]
[1 2 3 4 5 6]
[ 1 -1  1 -1  1 -1]


In [1]:
import numpy as np

list1 = np.array([1,2,3])

In [ ]:
print(list1[:3])

[1 2]


In [10]:
print(arr[:])

[1 2 3]


In [11]:
df = {}

In [20]:
df = {}
df["ss"] += 1
df

KeyError: 'ss'